# ALGORITMO SIMPLE DE REVERSIONES FUTUROS - BINANCE

In [14]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
from pathlib import Path
import pandas as pd
import numpy as np
import ta
import joblib
import json

import time
from datetime import datetime, timezone, timedelta

import winsound

In [15]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"
INTERVAL = "15m"                 # coherente con scanner 1H
LIMIT = 120                     # ~15 días de contexto
LOOKBACK = 120
MAX_SYMBOLS_TRAIN = 541       # limitar universo de entrenamiento
MIN_NOTIONAL_VOLUME = 10_000  # Sugerido 5_000_000

AVG_LIMIT_LONG = 25    # límite de KPI AVG para reversión LONG 
AVG_LIMIT_SHORT = 70    # límite de KPI AVG para reversión SHORT
AVG_MAX_ADVERSE = 0.9 # Límite de retroceso porcentual promedio con respecto a la reversión

FEATURES = [
    "rsi",
    "mfi",
    "stoch_k",
    "stoch_d",
    "macd_hist",
    "vol_ratio",
    "atr"
]

FILEPATH_ENTRY = "C:/Users/Lenovo S540/Python_scripts/ListaSimbolosReversionesFuturos.txt"
FILEPATH_EXIT = "C:/Users/Lenovo S540/Python_scripts/trades.json"

## 1. Búsqueda de Símbolos para reversión

In [16]:
# =========================================================
# 1️⃣ UNIVERSO BINANCE FUTURES
# =========================================================
def get_all_usdt_futures_symbols():
    """
    Retorna todos los símbolos de futuros perpetuos USDT en Binance.
    """
    response = requests.get(
        f"{BINANCE_FUTURES_BASE}/fapi/v1/exchangeInfo",
        timeout=10
    )
    response.raise_for_status()

    data = response.json()

    return [
        s["symbol"] for s in data["symbols"]
        if s["contractType"] == "PERPETUAL"
        and s["quoteAsset"] == "USDT"
        and s["status"] == "TRADING"
    ]


# =========================================================
# 1️⃣.1 FILTRANDO UNIVERSO BINANCE FUTURES PARA OBTENER LOS SÍMBOLOS ESTABLES, LÍQUIDOS Y REPRESENTATIVOS PARA ENTRENAMIENTO DEL MODELO
# =========================================================
def filter_symbols_pre_ml(
    symbols,
    interval="1h",
    lookback=LOOKBACK,
    min_notional_volume=MIN_NOTIONAL_VOLUME,
    min_atr_rel=0.0015,
    max_atr_rel=0.02,
    max_symbols=MAX_SYMBOLS_TRAIN
):
    """
    Filtra símbolos antes del scanner ML.
    Selecciona criptos líquidas, volátiles y operables
    para reversiones duraderas.
    """

    qualified = []

    for symbol in symbols:
        try:
            df = get_ohlc(symbol, interval=interval, limit=lookback)
            if df is None or len(df) < lookback:
                continue

            # --- Volumen nocional ---
            df["notional"] = df["close"] * df["volume"]
            avg_notional = df["notional"].mean()

            if avg_notional < min_notional_volume:
                continue

            # --- ATR relativo ---
            df["atr"] = ta.volatility.average_true_range(
                df["high"], df["low"], df["close"], window=14
            )
            atr_rel = (df["atr"] / df["close"]).mean()

            if not (min_atr_rel <= atr_rel <= max_atr_rel):
                continue

            # --- Estabilidad (evita pumps aislados) ---
            avg_range = (df["high"] - df["low"]).mean()
            range_std = (df["high"] - df["low"]).std()

            if range_std / avg_range > 1.8:
                continue

            qualified.append({
                "Symbol": symbol,
                "ATR_rel": round(atr_rel, 5),
                "Avg_Notional": int(avg_notional)
            })

        except Exception:
            continue

    if not qualified:
        return []

    dfq = pd.DataFrame(qualified)

    # Ranking por calidad operativa
    dfq = dfq.sort_values(
        ["Avg_Notional", "ATR_rel"],
        ascending=[False, False]
    )

    return dfq.head(max_symbols)["Symbol"].tolist()




# =========================================================
# 2️⃣ OHLC
# =========================================================
def get_ohlc(
    symbol: str,
    interval: str = "1h",
    limit: int = 500,
    max_total: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si max_total > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    max_total = max_total or limit

    while len(all_data) < max_total:
        fetch = min(1500, max_total - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < 50:
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["time", "open", "high", "low", "close", "volume"]]
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < 50:
        return None

    return df

# =========================================================
# 3️⃣ INDICADORES TÉCNICOS (SIMÉTRICOS LONG/SHORT)
# =========================================================

def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula indicadores técnicos para reversión y timing.
    Válida para scan_market_ml (1h) y confirm_entry_15m (15m).
    """

    # Momentum
    df["rsi"] = ta.momentum.rsi(df["close"], window=14)
    df["mfi"] = ta.volume.money_flow_index(
        df["high"], df["low"], df["close"], df["volume"], window=14
    )

    # Stochastic RSI
    stoch = ta.momentum.StochRSIIndicator(df["close"], window=14)
    df["stoch_k"] = stoch.stochrsi_k()
    df["stoch_d"] = stoch.stochrsi_d()

    # MACD Histogram
    macd = ta.trend.MACD(df["close"])
    df["macd_hist"] = macd.macd_diff()

    # Trend context
    df["ema50"] = ta.trend.ema_indicator(df["close"], window=50)
    df["dist_ema50"] = (df["close"] - df["ema50"]) / df["close"]

    # Volatility
    df["atr"] = ta.volatility.average_true_range(
        df["high"], df["low"], df["close"], window=14
    )
    #df["atr_rel"] = df["atr"] / df["close"]
    df["atr_rel"] = df["atr"] / df["atr"].rolling(50).mean()


    # Candle structure
    df["wick_ratio"] = (df["high"] - df["close"]) / (
        (df["high"] - df["low"]) + 1e-9
    )

    # Volume confirmation
    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean()

    # Limpieza mínima
    df = df.dropna().reset_index(drop=True)

    return df


    

# =========================================================
# 3️⃣.1 INDICADORES DE IMPULSO Y RETROCESO (SIMÉTRICOS LONG/SHORT)
# =========================================================

def compute_mae_atr(
    symbol: str,
    direction: str,
    interval: str = INTERVAL,
    lookback_bars: int = 64,
    atr_period: int = 14
):
    """
    Calcula el MAE (Maximum Adverse Excursion) reciente en unidades de ATR.
    Diseñado para ranking post-scan y validación de trades.

    Retorna:
        dict {
            "mae_atr": float,
            "avg_adverse": float,
            "max_adverse": float,
            "atr": float
        }
        o None si no es confiable
    """

    try:
        df = get_ohlc(symbol, interval=interval, limit=lookback_bars + atr_period + 5)
        if df is None or len(df) < atr_period + 10:
            return None

        df = df.sort_index()

        # ==========================
        # ATR
        # ==========================
        high = df["high"]
        low = df["low"]
        close = df["close"]
        prev_close = close.shift(1)

        tr = (
            (high - low)
            .combine((high - prev_close).abs(), max)
            .combine((low - prev_close).abs(), max)
        )

        atr_series = tr.rolling(atr_period).mean()
        atr = atr_series.iloc[-1]

        if atr is None or atr <= 0:
            return None

        # ==========================
        # Cálculo MAE
        # ==========================
        adverse_moves = []

        for i in range(len(df) - 1):
            entry = close.iloc[i]

            if direction == "LONG":
                adverse = (entry - low.iloc[i + 1]) / atr
            else:
                adverse = (high.iloc[i + 1] - entry) / atr

            if adverse >= 0:
                adverse_moves.append(adverse)

        if len(adverse_moves) < 10:
            return None

        avg_adverse = float(np.mean(adverse_moves))
        max_adverse = float(np.percentile(adverse_moves, 90))

        return {
            "mae_atr": avg_adverse,
            "avg_adverse": avg_adverse,
            "max_adverse": max_adverse,
        }

    except Exception as e:
        print(f"[WARN] compute_mae_atr {symbol}: {e}")
        return {
            "mae_atr": np.nan,
            "avg_adverse": np.nan,
            "max_adverse": np.nan,
        }



# ==========================
# Guardar JSON para monitoreo
# ==========================
def save_symbol_map(filepath: str, symbol_map: dict):
    """
    Guarda el diccionario actualizado en formato JSON.
    """
    try:
        with open(filepath, "w") as f:
            json.dump(symbol_map, f, indent=4)
            print(f"Lista de monitoreo actualizada el {datetime.now()}: {symbol_map.keys()}\nDisponible en la ruta: {filepath}")
    except Exception as e:
        print(f"[WARN] Error guardando JSON: {e}")



# =====================================
# Cargar JSON para monitoreo de entrada
# =====================================
def load_symbol_map(filepath: str) -> dict:
    """
    Carga archivo JSON con formato:
    {
        "ETHUSDT": "LONG",
        "LTCUSDT": "SHORT"
    }
    """
    try:
        path = Path(filepath)

        if not path.exists():
            return {}

        with open(path, "r") as f:
            data = json.load(f)

        # Validación básica
        valid = {}
        for k, v in data.items():
            if v in ["LONG", "SHORT"]:
                valid[k] = v

        return valid

    except Exception as e:
        print(f"[WARN] Error cargando JSON: {e}")
        return {}



# =====================================
# Cargar JSON para monitoreo de salida
# =====================================
def load_trades_map(filepath: str) -> dict:
    """
    Carga archivo JSON con formato:
    {
        "ETHUSDT": "LONG",
        "LTCUSDT": "SHORT"
    }
    """
    try:
        path = Path(filepath)

        if not path.exists():
            print(f"Ruta de acceso incorrecta")
            return {}

        with open(path, "r") as f:
            data = json.load(f)

        # Validación básica
        # valid = {}
        # for k, v in data.items():
        #     if v in ["LONG", "SHORT"]:
        #         valid[k] = v

        # return valid
        return data
    except Exception as e:
        print(f"[WARN] Error cargando JSON: {e}")
        return {}



# =====================================
# Actualizar JSON para monitoreo de salida
# =====================================
def update_trades_file(
    symbol: str,
    side: str,
    entry_price: float,
    target_usd_tp_partial: float,
    tp_partial_mult: float,
    sl_mult: float,
    max_leverage: int,
    min_capital: float,
    max_capital: float,
    max_risk_usd: float,
    opened_at_utc: str,
    end_time_utc: str,
    filepath_exit: str,
    initialize: bool
):

    """AGREGA UN SÍMBOLO NUEVO AL ARCHIVO DE MONITOREO DE TRADES"""
    
    symbol_entry = {
        symbol: {
            "symbol": symbol,
            "side": side,
            "status": "PENDING",
            "entry_price": entry_price,
            "target_usd_tp_partial": target_usd_tp_partial,
            "tp_partial_mult": tp_partial_mult,
            "sl_mult": sl_mult,
            "max_leverage": max_leverage,
            "min_capital": min_capital,
            "max_capital": max_capital,
            "max_risk_usd": max_risk_usd,
            "opened_at_utc": opened_at_utc,
            "end_time_utc": end_time_utc,
            "printed": "NO"
        }
    }

    if initialize:
        symbol_map = {}
    else:
        symbol_map = load_trades_map(FILEPATH_EXIT)
        
    symbol_map.update(symbol_entry)
    save_symbol_map(FILEPATH_EXIT, symbol_map)



# =====================================
# Alarma Local de Advertencia de Búsqueda
# =====================================
def local_warning():

    print("Oportunidades de reversión encontradas!")

    # Frecuencias en la 4ta y 5ta octava (Hz) para que se note el "punch"
    E4 = 330   # Mi
    G4 = 392   # Sol
    A4 = 440   # La
    B4 = 494   # Si
    
    # Tiempos (ms) ajustados para el groove de Oasis
    NEGRA = 420
    CORCHEA = 210
    SEMI = 105
    SILENCIO = 0.03
    
    
    # Frase 1: El ataque (E-E-G-E)
    winsound.Beep(E4, CORCHEA)
    winsound.Beep(E4, CORCHEA)
    winsound.Beep(G4, SEMI)
    winsound.Beep(E4, SEMI)
    winsound.Beep(G4, SEMI)
    
    time.sleep(SILENCIO)
    
    # Frase 2: El riff descendente (B-A)
    winsound.Beep(B4, CORCHEA)
    winsound.Beep(B4, SEMI)
    
    # Frase 3: Respuesta (A-B) y cierre con Sol
    winsound.Beep(A4, SEMI)
    winsound.Beep(E4, SEMI)
    winsound.Beep(B4, CORCHEA)

In [28]:
# ==========================================================
# 1️⃣ OBTENER UNIVERSO DE SÍMBOLOS
# ==========================================================
all_symbols = get_all_usdt_futures_symbols()
print(f"Total símbolos USDT Futures: {len(all_symbols)}")

# ==========================================================
# 2️⃣ FILTRADO PREVIO (LIQUIDEZ + VOLATILIDAD)
# ==========================================================
# symbols = filter_symbols_pre_ml(
#     symbols=all_symbols,
#     interval="1h",
#     max_symbols=MAX_SYMBOLS_TRAIN
# )

# symbols = symbols[:MAX_SYMBOLS_TRAIN]

# print(f"Símbolos seleccionados para análisis: {len(symbols)}")

rows = []


for sym in all_symbols:
    try:
        # 1️⃣ Descargar datos históricos
        df = get_ohlc(sym, interval=INTERVAL, limit=50)
        df = add_indicators(df)

        row = df.iloc[-1]
        rows.append({
                    "Timestamp": datetime.now(),
                    "Symbol": sym,
                    **{f: row[f] for f in FEATURES},
                    #"label": label,
                    # "ATR": atr,
                    # "ATR_rel": atr / entry
                })

    except Exception as e:
        print(f"[WARN] Error creando dataset para {sym}: {e}")
        continue

df_out = pd.DataFrame(rows)
df_out.reset_index(drop=True, inplace=True)
df_out["KPI_avg"] = (df_out["rsi"] + (df_out["stoch_k"]*100) + (df_out["stoch_d"]*100) + df_out["mfi"])/4
df_out["KPI_avg2"] = (df_out["rsi"] + df_out["mfi"])/2

df_out['Direction'] = df_out['KPI_avg'].apply(lambda x: 'SHORT' if x >= AVG_LIMIT_SHORT else 'NONE')
df_out['Direction'] = df_out['KPI_avg'].apply(lambda x: 'LONG' if x <= AVG_LIMIT_LONG else 'NONE')
df_out = df_out[df_out["Direction"]!="NONE"]

df_out['mae_atr'] = df_out.apply(lambda row: compute_mae_atr(row['Symbol'], row['Direction']).get('mae_atr'), axis=1)
df_out['avg_adverse'] = df_out.apply(lambda row: compute_mae_atr(row['Symbol'], row['Direction'])['avg_adverse'], axis=1)
df_out['max_adverse'] = df_out.apply(lambda row: compute_mae_atr(row['Symbol'], row['Direction'])['max_adverse'], axis=1)

# Cálculo de Score 1 ponderado de indicadores con magnitud de reversión
comp_mae = (1 - df_out['mae_atr']) * 100
comp_kpi_short = df_out['KPI_avg']
comp_kpi_long = 100 - df_out['KPI_avg']

df_out['SCORE'] = np.where(
    df_out['Direction'] == 'SHORT',
    (comp_kpi_short + comp_mae) / 2,
    (comp_kpi_long + comp_mae) / 2
)

# Cálculo de Score 2 ponderado de indicadores con magnitud de reversión
comp_mae = (1 - df_out['mae_atr']) * 100
comp_kpi_short = df_out['KPI_avg2']
comp_kpi_long = 100 - df_out['KPI_avg2']

df_out['SCORE2'] = np.where(
    df_out['Direction'] == 'SHORT',
    (comp_kpi_short + comp_mae) / 2,
    (comp_kpi_long + comp_mae) / 2
)

df_out = df_out[(df_out['KPI_avg2'] <= AVG_LIMIT_LONG) | (df_out['KPI_avg2'] >= AVG_LIMIT_SHORT)]
df_out = df_out[df_out['mae_atr'] <= AVG_MAX_ADVERSE]

if len(df_out) > 0:
    local_warning()
else:
    print("No hay símbolos con patrón de reversión por el momento.")

pd.set_option("display.max_rows", 200)
#df_out = df_out.sort_values(by=['KPI_avg'], ascending=[True])
df_out = df_out.sort_values(by=['SCORE2'], ascending=[False])
df_out = df_out.reset_index(drop=True)
df_out.style.set_properties(**{'background-color': 'green'}, subset=['KPI_avg2', 'mae_atr', 'rsi', 'mfi'])

Total símbolos USDT Futures: 544
Oportunidades de reversión encontradas!


,Timestamp,Symbol,rsi,mfi,stoch_k,stoch_d,macd_hist,vol_ratio,atr,KPI_avg,KPI_avg2,Direction,mae_atr,avg_adverse,max_adverse,SCORE,SCORE2
0,2026-03-05 18:30:26.497761,CHILLGUYUSDT,23.102714,12.735599,0.112327,0.085277,-0.000021,0.002911,0.000105,13.899673,17.919157,LONG,0.656033,0.656033,1.430657,60.248505,58.238764
1,2026-03-05 18:31:01.090133,HOMEUSDT,26.680371,4.941492,0.097696,0.051245,-0.000044,0.056901,0.000193,11.628992,15.810931,LONG,0.682276,0.682276,1.133485,60.071681,57.980711
2,2026-03-05 18:31:28.867547,LABUSDT,30.257620,16.977193,0.112881,0.191129,0.000329,0.111201,0.004251,19.408955,23.617407,LONG,0.623192,0.623192,1.036657,59.135943,57.031717
3,2026-03-05 18:29:55.288946,RIFUSDT,34.332696,15.415151,0.139979,0.126992,-0.000052,0.523838,0.000227,19.111236,24.873923,LONG,0.769921,0.769921,1.604743,51.948310,49.066967
4,2026-03-05 18:30:34.608030,COOKIEUSDT,33.719255,13.655737,0.116273,0.134388,-0.000000,0.005063,0.000168,18.110256,23.687496,LONG,0.891746,0.884953,2.063507,46.357551,43.568932


In [18]:
# pd.set_option("display.max_rows", 200)
# df_out = df_out.sort_values(by=['KPI_avg', 'mae_atr'], ascending=[False, True])
# df_out

In [19]:
# df_out = df_out.sort_values(by=['rsi', 'stoch_k', 'stoch_d', 'mfi'], ascending=[False, False, False, False])
# df_out

In [7]:
# df_out = df_out.sort_values(by=['rsi', 'stoch_k', 'stoch_d', 'mfi'], ascending=[True, True, True, True])
# df_out

## 2. Lista de Símbolos para Monitoreo de Entrada

### Guardar JSON para Monitoreo

In [29]:
#symbol_list = [0]
# symbol_list = range(3)
symbol_list = range(len(df_out))
df_json = df_out.loc[symbol_list]
symbol_map = dict(zip(df_json["Symbol"], df_json["Direction"]))

save_symbol_map(FILEPATH_ENTRY, symbol_map)

Lista de monitoreo actualizada el 2026-03-05 18:36:06.552780: dict_keys(['CHILLGUYUSDT', 'HOMEUSDT', 'LABUSDT', 'RIFUSDT', 'COOKIEUSDT'])
Disponible en la ruta: C:/Users/Lenovo S540/Python_scripts/ListaSimbolosReversionesFuturos.txt


### Agregar nuevos Símbolos a JSON guardado para Monitoreo de Entrada

In [23]:
# additional_symbols = [10, 12, 14]

# symbol_map = load_symbol_map(FILEPATH_ENTRY)

# df_additional = df_out.loc[additional_symbols]
# additional_map = dict(zip(df_additional["Symbol"], df_additional["Direction"]))

# symbol_map.update(additional_map)
# save_symbol_map(FILEPATH_ENTRY, symbol_map)

In [ ]:
additional_map = {"VVVUSDT": "LONG",
#                   "BTCUSDT": "SHORT",
#                   "SOLUSDT": "SHORT",
                   # "NEARUSDT": "SHORT"
                }

symbol_map = load_symbol_map(FILEPATH_ENTRY)
# symbol_map = {}
symbol_map.update(additional_map)
save_symbol_map(FILEPATH_ENTRY, symbol_map)

## 3. Lista de Símbolos para Monitoreo de Salida

### Agregar nuevos Símbolos a JSON guardado para Monitoreo de Salida

In [28]:
print(symbol_map)

{'FOGOUSDT': 'LONG', 'COAIUSDT': 'LONG', '1000WHYUSDT': 'LONG', 'TACUSDT': 'LONG', 'ANKRUSDT': 'LONG', 'BREVUSDT': 'LONG', 'EDENUSDT': 'LONG', 'BSVUSDT': 'LONG', 'SPORTFUNUSDT': 'LONG', 'HEIUSDT': 'LONG', 'TSTUSDT': 'LONG'}


In [26]:
# ==========================================================
# SÍMBOLO Y MOMENTO DE ENTRADA
# ==========================================================
initialize = True #True para eliminar todos los símbolos existentes en json actual, False para mantener los existentes y agregar los nuevos
symbol = "VVV"
side = "LONG"
time_entry = "17:50" # hora local Colombia

symbol = symbol + "USDT"
date = str(datetime.now())[:10]
opened_at_utc = date + "T" + time_entry + ":00+00:00"
format_pattern = "%Y-%m-%dT%H:%M:%S+00:00"
opened_at_utc_formatted = datetime.strptime(opened_at_utc, format_pattern) + timedelta(hours=5)
opened_at_utc = opened_at_utc_formatted.strftime(format_pattern)
end_time_utc_formatted = opened_at_utc_formatted + timedelta(hours=3)
end_time_utc = end_time_utc_formatted.strftime(format_pattern)

# opened_at_utc = "2026-02-13T14:00:00+00:00"
# end_time_utc = "2026-02-13T18:00:00+00:00"


# ==========================================================
# PRECIO DE ENTRADA
# ==========================================================
last_price = get_ohlc(symbol=symbol, interval="1m", limit=50)
last_price = last_price.iloc[-1, -2]
entry_price = last_price

# entry_price = 0.18933 # Precio de entrada REAL (ejecutado por ti)


# ==========================================================
# PARÁMETROS DE GESTIÓN DEL RIESGO
# ==========================================================
target_usd_tp_partial = 25.0
max_leverage = 20
min_capital = 120.0
max_capital = 300.0
max_risk_usd = 70.0

tp_partial_mult = 1.2
sl_mult = 0.8

In [27]:
update_trades_file(
    symbol=symbol,
    side=side,
    entry_price=entry_price,
    target_usd_tp_partial=target_usd_tp_partial,
    tp_partial_mult=tp_partial_mult,
    sl_mult=sl_mult,
    max_leverage=max_leverage,
    min_capital=min_capital,
    max_capital=max_capital,
    max_risk_usd=max_risk_usd,
    opened_at_utc=opened_at_utc,
    end_time_utc=end_time_utc,
    filepath_exit=FILEPATH_EXIT,
    initialize=initialize
)

Lista de monitoreo actualizada el 2026-03-05 17:58:51.960560: dict_keys(['VVVUSDT'])
Disponible en la ruta: C:/Users/Lenovo S540/Python_scripts/trades.json
